# Getting reliable JSON out of an LLM

**The Agent Build Log · Build #004 · GenAI & LLM Foundations 4/8**

[Build #003](../03-first-llm-api-call/) made one call production-ready. Most real apps don't want prose back,
they want **data**: fields your code can store, route and act on. This build extracts 6 fields from
20 messy support tickets, three ways, and measures how often each one gives you something your code can use.

1. [The task and the schema](#1-the-task-and-the-schema)
2. [Approach A: "Reply in JSON"](#2-approach-a-reply-in-json)
3. [Approach B: schema in the prompt + one retry](#3-approach-b-schema-in-the-prompt--one-retry)
4. [Approach C: structured outputs](#4-approach-c-structured-outputs)
5. [Side by side](#5-side-by-side)
6. [Valid JSON is not the same as correct data](#6-valid-json-is-not-the-same-as-correct-data)
7. [Findings](#7-findings)

**Needs:** an Anthropic API key in a `.env` file. Uses **Claude Haiku 4.5**; a full run costs a few cents.

In [1]:
import json, os, re, statistics, time
from pathlib import Path
from typing import Literal, Optional

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, ValidationError
import anthropic

load_dotenv(Path.cwd() / ".env") or load_dotenv(Path.cwd().parents[1] / ".env")
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in a .env file (see README)"
pd.set_option("display.width", 150); pd.set_option("display.max_colwidth", 80)

MODEL = "claude-haiku-4-5"
PRICE = {"input": 1.00, "output": 5.00}   # $ per 1M tokens, Claude Haiku 4.5
client = anthropic.Anthropic()
DATA = json.loads(Path("data/tickets.json").read_text())
TICKETS = DATA["tickets"]
print(f"{len(TICKETS)} tickets | model: {MODEL} | anthropic SDK {anthropic.__version__}")

20 tickets | model: claude-haiku-4-5 | anthropic SDK 1.11.0


## 1. The task and the schema

Six fields per ticket. Three are **objective** (there's one right answer): order id, refund amount, email.
Three are **judgement calls** (category, priority, sentiment): the labels in `data/tickets.json` are mine, so
for those we report *agreement with my labels*, not accuracy.

In [2]:
class Ticket(BaseModel):
    category: Literal["billing", "bug", "account", "shipping", "feature_request"]
    priority: Literal["low", "medium", "high", "urgent"]
    order_id: Optional[str]          # e.g. "ORD-48213", or null
    refund_amount: Optional[float]   # in the ticket's currency, or null
    customer_email: Optional[str]
    sentiment: Literal["negative", "neutral", "positive"]

OBJECTIVE = ["order_id", "refund_amount", "customer_email"]
JUDGEMENT = ["category", "priority", "sentiment"]
print(json.dumps(Ticket.model_json_schema(), indent=1)[:600], "...")

{
 "properties": {
  "category": {
   "enum": [
    "billing",
    "bug",
    "account",
    "shipping",
    "feature_request"
   ],
   "title": "Category",
   "type": "string"
  },
  "priority": {
   "enum": [
    "low",
    "medium",
    "high",
    "urgent"
   ],
   "title": "Priority",
   "type": "string"
  },
  "order_id": {
   "anyOf": [
    {
     "type": "string"
    },
    {
     "type": "null"
    }
   ],
   "title": "Order Id"
  },
  "refund_amount": {
   "anyOf": [
    {
     "type": "number"
    },
    {
     "type": "null"
    }
   ],
   "title": "Refund Amount"
  },
  "customer_ ...


In [3]:
def cost(usage) -> float:
    return (usage.input_tokens * PRICE["input"] + usage.output_tokens * PRICE["output"]) / 1e6

def text_of(msg) -> str:
    return "".join(b.text for b in msg.content if b.type == "text")

def try_parse(text: str):
    """Strict: does the raw text parse as JSON, and does it match the schema?"""
    try:
        obj = json.loads(text)
    except json.JSONDecodeError as e:
        return None, None, f"not JSON: {str(e)[:40]}"
    try:
        return obj, Ticket.model_validate(obj), None
    except ValidationError as e:
        return obj, None, "schema: " + "; ".join(f"{'.'.join(map(str, x['loc']))} {x['type']}" for x in e.errors()[:3])

def salvage(text: str) -> str:
    """The usual hack: strip code fences / prose and keep the first {...} block."""
    m = re.search(r"\{.*\}", text, re.S)
    return m.group(0) if m else text

def norm(v):
    if isinstance(v, str):
        v = v.strip().lower()
        return v or None
    return v

def field_matches(pred: Ticket, label: dict) -> dict:
    out = {}
    for f in OBJECTIVE + JUDGEMENT:
        p, l = norm(getattr(pred, f)), norm(label[f])
        if f == "refund_amount" and p is not None and l is not None:
            out[f] = abs(float(p) - float(l)) < 0.01
        else:
            out[f] = p == l
    return out

## 2. Approach A: "Reply in JSON"

What most people try first: describe the fields in words and ask for JSON.

In [4]:
PROMPT_A = ("Extract the category, priority, order id, refund amount, customer email and sentiment "
            "from this support ticket. Reply in JSON.\n\nTicket: {text}")

def run_a(t):
    t0 = time.perf_counter()
    m = client.messages.create(model=MODEL, max_tokens=500,
                               messages=[{"role": "user", "content": PROMPT_A.format(text=t["text"])}])
    raw = text_of(m)
    obj, ticket, err = try_parse(raw)
    _, salvaged, s_err = try_parse(salvage(raw))
    return {"id": t["id"], "raw": raw, "json ok": obj is not None, "schema ok": ticket is not None,
            "salvaged schema ok": salvaged is not None, "error": err, "salvage error": s_err,
            "fenced": raw.lstrip().startswith("```"), "ticket": ticket or salvaged,
            "seconds": time.perf_counter() - t0, "out tokens": m.usage.output_tokens, "cost": cost(m.usage)}

A = [run_a(t) for t in TICKETS]
print("Sample raw output (ticket 1):\n", A[0]["raw"][:400])
pd.DataFrame(A)[["id", "json ok", "schema ok", "salvaged schema ok", "fenced", "error", "salvage error"]]

Sample raw output (ticket 1):
 ```json
{
  "category": "Billing",
  "priority": "High",
  "order_id": "ORD-48213",
  "refund_amount": "$49.99",
  "customer_email": "jane.doe@example.com",
  "sentiment": "Neutral"
}
```


,id,json ok,schema ok,salvaged schema ok,fenced,error,salvage error
0,1,False,False,False,True,not JSON: Expecting value: line 1 column 1 (char 0,schema: category literal_error; priority literal_error; refund_amount float_...
1,2,False,False,False,True,not JSON: Expecting value: line 1 column 1 (char 0,schema: category literal_error; priority literal_error; refund_amount float_...
2,3,False,False,False,True,not JSON: Expecting value: line 1 column 1 (char 0,schema: category literal_error; priority literal_error; sentiment literal_error
3,4,False,False,False,True,not JSON: Expecting value: line 1 column 1 (char 0,schema: category literal_error; priority literal_error; sentiment literal_error
4,5,False,False,False,True,not JSON: Expecting value: line 1 column 1 (char 0,schema: category literal_error; priority literal_error; sentiment literal_error
5,6,False,False,False,True,not JSON: Expecting value: line 1 column 1 (char 0,schema: category literal_error; priority literal_error; refund_amount float_...
6,7,False,False,False,True,not JSON: Expecting value: line 1 column 1 (char 0,schema: category literal_error; priority literal_error; sentiment literal_error
7,8,False,False,False,True,not JSON: Expecting value: line 1 column 1 (char 0,schema: category literal_error; priority literal_error; sentiment literal_error
8,9,False,False,False,True,not JSON: Expecting value: line 1 column 1 (char 0,schema: category literal_error; priority literal_error; sentiment literal_error
9,10,False,False,False,True,not JSON: Expecting value: line 1 column 1 (char 0,schema: category literal_error; priority literal_error; sentiment literal_error


## 3. Approach B: schema in the prompt + one retry

Paste the exact JSON schema into the system prompt, ask for JSON only, and if parsing or validation fails,
send the error back once and ask again.

In [5]:
SYSTEM_B = ("You extract structured data from support tickets. Return ONLY a JSON object that matches this "
            "JSON schema exactly. No prose, no code fences.\n\n" + json.dumps(Ticket.model_json_schema()))

def run_b(t):
    t0 = time.perf_counter(); calls = 0; out_tok = 0; total_cost = 0.0
    messages = [{"role": "user", "content": t["text"]}]
    for attempt in range(2):
        m = client.messages.create(model=MODEL, max_tokens=500, system=SYSTEM_B, messages=messages)
        calls += 1; out_tok += m.usage.output_tokens; total_cost += cost(m.usage)
        raw = text_of(m); obj, ticket, err = try_parse(raw)
        if attempt == 0:
            first_ok, first_err, first_raw = ticket is not None, err, raw
            first_seconds, first_out, first_cost = time.perf_counter() - t0, m.usage.output_tokens, cost(m.usage)
            _, first_salvaged, _ = try_parse(salvage(raw))   # what a careful engineer would ship
        if ticket is not None:
            break
        messages += [{"role": "assistant", "content": raw},
                     {"role": "user", "content": f"That was not valid ({err}). Return only the corrected JSON object."}]
    return {"id": t["id"], "raw": first_raw, "first try ok": first_ok, "first error": first_err,
            "first fenced": first_raw.lstrip().startswith("```"), "final raw": raw, "final error": err,
            "ok after retry": ticket is not None, "calls": calls, "ticket": ticket,
            "salvaged first try ok": first_salvaged is not None, "salvaged ticket": first_salvaged,
            "first seconds": first_seconds, "first out tokens": first_out, "first cost": first_cost,
            "seconds": time.perf_counter() - t0, "out tokens": out_tok, "cost": total_cost}

B = [run_b(t) for t in TICKETS]
print("Sample first-try output (ticket 1):"); print(B[0]["raw"][:300])
pd.DataFrame(B)[["id", "first try ok", "first fenced", "ok after retry", "final error"]]

Sample first-try output (ticket 1):
```json
{
  "category": "billing",
  "priority": "high",
  "order_id": "ORD-48213",
  "refund_amount": 49.99,
  "customer_email": "jane.doe@example.com",
  "sentiment": "neutral"
}
```


,id,first try ok,first fenced,ok after retry,final error
0,1,False,True,False,not JSON: Expecting value: line 1 column 1 (char 0
1,2,False,True,True,None
2,3,False,True,False,not JSON: Expecting value: line 1 column 1 (char 0
3,4,False,True,True,None
4,5,False,True,False,not JSON: Expecting value: line 1 column 1 (char 0
5,6,False,True,True,None
6,7,False,True,False,not JSON: Expecting value: line 1 column 1 (char 0
7,8,False,True,True,None
8,9,False,True,True,None
9,10,False,True,False,not JSON: Expecting value: line 1 column 1 (char 0


## 4. Approach C: structured outputs

The API constrains generation to the schema. With the Python SDK you pass the Pydantic model itself and get a
validated object back. New schemas have a one-time compile cost (then a 24-hour cache), so we time the first
call of this run separately.

In [6]:
def run_c(t):
    t0 = time.perf_counter()
    m = client.messages.parse(model=MODEL, max_tokens=500, output_format=Ticket,
                              messages=[{"role": "user", "content": "Extract the fields from this support ticket.\n\n" + t["text"]}])
    ticket = m.parsed_output
    return {"id": t["id"], "raw": text_of(m), "schema ok": ticket is not None, "stop_reason": m.stop_reason,
            "ticket": ticket, "seconds": time.perf_counter() - t0, "out tokens": m.usage.output_tokens, "cost": cost(m.usage)}

Cres = [run_c(t) for t in TICKETS]
print("Sample raw output (ticket 1):", Cres[0]["raw"])
print(f"first call of this run: {Cres[0]['seconds']:.2f}s | median of the rest: {statistics.median(r['seconds'] for r in Cres[1:]):.2f}s")
pd.DataFrame(Cres)[["id", "schema ok", "stop_reason", "seconds"]]

Sample raw output (ticket 1): {"category": "billing", "priority": "high", "order_id": "ORD-48213", "refund_amount": 49.99, "customer_email": "jane.doe@example.com", "sentiment": "neutral"}
first call of this run: 1.98s | median of the rest: 0.93s


,id,schema ok,stop_reason,seconds
0,1,True,end_turn,1.975653
1,2,True,end_turn,0.871632
2,3,True,end_turn,0.944235
3,4,True,end_turn,0.817564
4,5,True,end_turn,0.747215
5,6,True,end_turn,0.831148
6,7,True,end_turn,0.934063
7,8,True,end_turn,0.914378
8,9,True,end_turn,0.745637
9,10,True,end_turn,1.957581


## 5. Side by side

In [7]:
def summary(name, rows, ok_key):
    ok = sum(r[ok_key] for r in rows)
    return {"approach": name, "usable (schema-valid)": f"{ok}/{len(rows)}", "usable %": round(ok / len(rows) * 100),
            "API calls": sum(r.get("calls", 1) for r in rows),
            "median s / ticket": round(statistics.median(r["seconds"] for r in rows), 2),
            "avg out tokens": round(statistics.mean(r["out tokens"] for r in rows)),
            "cost $ (20 tickets)": round(sum(r["cost"] for r in rows), 4)}

side = pd.DataFrame([
    summary('A: "reply in JSON" (strict parse)', A, "schema ok"),
    {**summary('A + salvage hack (strip fences, grab {...})', A, "salvaged schema ok"), "API calls": "same calls"},
    summary("B: schema in prompt, first try", B, "first try ok"),
    summary("B: schema in prompt + 1 retry", B, "ok after retry"),
    summary("B: schema in prompt + salvage hack, first try",
            [{**r, "seconds": r["first seconds"], "out tokens": r["first out tokens"], "cost": r["first cost"], "calls": 1} for r in B],
            "salvaged first try ok"),
    summary("C: structured outputs", Cres, "schema ok"),
])
fail_reasons = pd.Series([r["error"] for r in A if r["error"]]).str.split(":").str[0].value_counts().to_dict()
print("Approach A strict-parse failures by reason:", fail_reasons, "| fenced outputs:", sum(r["fenced"] for r in A))
side

Approach A strict-parse failures by reason: {'not JSON': 20} | fenced outputs: 20


,approach,usable (schema-valid),usable %,API calls,median s / ticket,avg out tokens,cost $ (20 tickets)
0,"A: ""reply in JSON"" (strict parse)",0/20,0,20,0.89,73,0.0086
1,"A + salvage hack (strip fences, grab {...})",0/20,0,same calls,0.89,73,0.0086
2,"B: schema in prompt, first try",0/20,0,40,1.53,125,0.0267
3,B: schema in prompt + 1 retry,10/20,50,40,1.53,125,0.0267
4,"B: schema in prompt + salvage hack, first try",20/20,100,20,0.81,68,0.0128
5,C: structured outputs,20/20,100,20,0.94,47,0.0157


## 6. Valid JSON is not the same as correct data

Structured outputs guarantee the **shape**. They don't guarantee the **values**. Here's how the usable outputs
compare against the labels, field by field.

In [8]:
def accuracy(rows):
    usable = [r for r in rows if r["ticket"] is not None]
    per = {f: 0 for f in OBJECTIVE + JUDGEMENT}
    wrong = []
    for r in usable:
        label = next(t["label"] for t in TICKETS if t["id"] == r["id"])
        m = field_matches(r["ticket"], label)
        for f, ok in m.items():
            per[f] += ok
            if not ok and f in OBJECTIVE:
                wrong.append({"id": r["id"], "field": f, "got": getattr(r["ticket"], f), "expected": label[f]})
    n = len(usable)
    return n, {f: f"{per[f]}/{n}" for f in per}, wrong

acc_rows, wrong_c = [], []
B_salv = [{**r, "ticket": r["salvaged ticket"]} for r in B]
for name, rows in [("A + salvage", A), ("B + retry", B), ("B + salvage (first try)", B_salv), ("C: structured outputs", Cres)]:
    n, per, wrong = accuracy(rows)
    acc_rows.append({"approach": name, "usable": n, **per})
    if name.startswith("C"): wrong_c = wrong
acc = pd.DataFrame(acc_rows)
print("Objective-field mistakes in C (valid JSON, wrong value):")
print(pd.DataFrame(wrong_c).to_string(index=False) if wrong_c else "  none")
acc

Objective-field mistakes in C (valid JSON, wrong value):
  none


,approach,usable,order_id,refund_amount,customer_email,category,priority,sentiment
0,A + salvage,0,0/0,0/0,0/0,0/0,0/0,0/0
1,B + retry,10,10/10,10/10,10/10,10/10,8/10,8/10
2,B + salvage (first try),20,20/20,20/20,20/20,20/20,17/20,16/20
3,C: structured outputs,20,20/20,20/20,20/20,20/20,18/20,14/20


## 7. Findings

Every number below comes from the cells above; these are the numbers the LinkedIn post uses.

In [9]:
f = {
    "model": MODEL, "sdk": anthropic.__version__, "tickets": len(TICKETS),
    "side_by_side": side.to_dict("records"),
    "a_fail_reasons": fail_reasons, "a_fenced": sum(r["fenced"] for r in A),
    "a_sample_raw": A[0]["raw"], "c_sample_raw": Cres[0]["raw"],
    "a_errors": [{"id": r["id"], "error": r["error"], "salvage_error": r["salvage error"]} for r in A if r["error"] or r["salvage error"]],
    "b_first_errors": [{"id": r["id"], "error": r["first error"]} for r in B if r["first error"]],
    "b_first_fenced": sum(r["first fenced"] for r in B), "b_sample_first_raw": B[0]["raw"],
    "b_final_failures": [{"id": r["id"], "error": r["final error"], "raw": r["final raw"][:300]} for r in B if not r["ok after retry"]],
    "c_first_call_s": round(Cres[0]["seconds"], 2), "c_median_rest_s": round(statistics.median(r["seconds"] for r in Cres[1:]), 2),
    "accuracy": acc.to_dict("records"), "c_objective_mistakes": wrong_c,
}
Path("findings.json").write_text(json.dumps(f, indent=2, ensure_ascii=False, default=str))
print(json.dumps({k: f[k] for k in ("side_by_side", "a_fail_reasons", "a_fenced", "accuracy", "c_objective_mistakes", "c_first_call_s", "c_median_rest_s")}, indent=1, default=str))

{
 "side_by_side": [
  {
   "approach": "A: \"reply in JSON\" (strict parse)",
   "usable (schema-valid)": "0/20",
   "usable %": 0,
   "API calls": 20,
   "median s / ticket": 0.89,
   "avg out tokens": 73,
   "cost $ (20 tickets)": 0.0086
  },
  {
   "approach": "A + salvage hack (strip fences, grab {...})",
   "usable (schema-valid)": "0/20",
   "usable %": 0,
   "API calls": "same calls",
   "median s / ticket": 0.89,
   "avg out tokens": 73,
   "cost $ (20 tickets)": 0.0086
  },
  {
   "approach": "B: schema in prompt, first try",
   "usable (schema-valid)": "0/20",
   "usable %": 0,
   "API calls": 40,
   "median s / ticket": 1.53,
   "avg out tokens": 125,
   "cost $ (20 tickets)": 0.0267
  },
  {
   "approach": "B: schema in prompt + 1 retry",
   "usable (schema-valid)": "10/20",
   "usable %": 50,
   "API calls": 40,
   "median s / ticket": 1.53,
   "avg out tokens": 125,
   "cost $ (20 tickets)": 0.0267
  },
  {
   "approach": "B: schema in prompt + salvage hack, first try",
